# 🕵️ Dia 09 — Sub-redes e CIDR para ler logs de firewall

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 9* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-09.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 14h00. O log do firewall da sede explica as `S0` perto do meio-dia: a TI bloqueou `203.0.113.77` no firewall antes do bloqueio de DNS. Mas há três linhas de `10.20.40.57`, que não é da sede: é a **Agência Centro**, a rede dos caixas.

Rafa: *"Anota, prioriza e não conclui nada ainda. E dois cooperados usaram o mesmo IP no internet banking. Me diz o que esse IP prova."*

Neste notebook você vai:

1. **Treinar a conta de cabeça** com o jogo do tamanho de bloco.
2. **Ver a pegadinha do `is_private`** com endereços de documentação.
3. **Ler o `fw-sede`**, converter BRT → UTC e cruzar com o `conn.log` do Dia 4.
4. **Classificar fluxos por segmento e direção** e gerar a lista de prioridades.
5. **Gerar o mapa de fluxos** em mermaid, com a seta prioritária em vermelho.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

## 1. 🎮 O jogo do tamanho de bloco

O método do original: ache o octeto onde o prefixo termina, calcule o tamanho do bloco (2 elevado aos bits de host **naquele octeto**) e arredonde para baixo. Responda primeiro e último endereço; o `ipaddress` confere.

`MODO_JOGO = False` só mostra as respostas.

In [ ]:
import ipaddress, random

def explicar(cidr):
    """Mostra o raciocínio do tamanho de bloco, passo a passo."""
    ip, pref = cidr.split("/")
    pref = int(pref)
    octeto = min(pref // 8, 3)                       # índice do octeto onde o prefixo "corta" (0 a 3)
    bits_host_no_octeto = 8 - (pref - 8 * octeto)    # quantos bits de host sobram naquele octeto
    bloco = 2 ** bits_host_no_octeto
    o = [int(x) for x in ip.split(".")]
    inicio = (o[octeto] // bloco) * bloco            # arredonda para baixo, no múltiplo do bloco
    return f"o prefixo /{pref} corta o {octeto + 1}º octeto; bloco de {bloco}; {o[octeto]} cai em {inicio}–{inicio + bloco - 1}"

MODO_JOGO = True
random.seed(9)                                       # semente fixa: todo mundo recebe as mesmas perguntas
PERGUNTAS = ["172.20.14.77/20", "10.20.34.19/22", "10.20.40.57/24"] + [
    f"10.{random.randint(0, 255)}.{random.randint(0, 255)}.{random.randint(1, 254)}/{random.choice([12, 18, 21, 26, 27, 29, 30])}"
    for _ in range(7)]

pontos = 0
for i, cidr in enumerate(PERGUNTAS, start=1):
    rede = ipaddress.ip_interface(cidr).network      # ip_interface aceita host + prefixo
    certo = (str(rede.network_address), str(rede.broadcast_address))
    print(f"\n[{i}/{len(PERGUNTAS)}] {cidr}")
    if MODO_JOGO:
        primeiro = input("   Primeiro endereço: ").strip()
        ultimo = input("   Último endereço: ").strip()
        ok = (primeiro, ultimo) == certo
        pontos += ok
        print("   ✅ Acertou!" if ok else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {certo[0]} a {certo[1]} ({rede.num_addresses} endereços) — {explicar(cidr)}")
if MODO_JOGO:
    print(f"\n🏁 {pontos}/{len(PERGUNTAS)}", "— Checkpoint BR cumprido. 🥷" if pontos >= 8 else "— meta: 8 de 10. Jogue de novo!")

## 2. A pegadinha do `is_private`

In [ ]:
RFC1918 = [ipaddress.ip_network(n) for n in ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16")]
CGNAT = ipaddress.ip_network("100.64.0.0/10")

for ip in ["10.20.40.57", "203.0.113.77", "198.51.100.230", "100.64.18.201", "172.31.255.255", "172.32.0.1"]:
    a = ipaddress.ip_address(ip)
    rfc = any(a in r for r in RFC1918)
    print(f"{ip:16} is_private={str(a.is_private):5}  RFC1918={str(rfc):5}  CGNAT={str(a in CGNAT):5}")
print("\n➡️ 203.0.113.77 é 'private' para o Python (documentação), mas NÃO é rede interna.")
print("   Teste a RFC 1918 explicitamente, como o script do original.")

In [ ]:
# ------------------------------------------------------------
# DADOS DO CASO (Operação Isca — tudo fictício)
# Primeiro tentamos baixar do repositório no GitHub; se não houver
# internet (ou o repo estiver fora do ar), usamos a cópia embutida.
# ------------------------------------------------------------
import urllib.request   # baixa arquivos pela web, sem instalar nada

BASE = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-09/"

# Cópias embutidas: o plano B. São idênticas aos arquivos do repo.
EMBUTIDO = {}
EMBUTIDO['ufw-fw-sede.log'] = 'Oct  5 11:47:33 fw-sede kernel: [UFW BLOCK] IN=eth1 OUT=eth0 MAC=52:54:00:1a:2b:3c:52:54:00:4d:5e:6f:08:00 SRC=10.20.36.21 DST=203.0.113.77 LEN=60 TOS=0x00 PREC=0x00 TTL=127 ID=31002 DF PROTO=TCP SPT=50990 DPT=443 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 11:47:34 fw-sede kernel: [UFW BLOCK] IN=eth1 OUT=eth0 MAC=52:54:00:1a:2b:3c:52:54:00:4d:5e:6f:08:00 SRC=10.20.36.21 DST=203.0.113.77 LEN=60 TOS=0x00 PREC=0x00 TTL=127 ID=31003 DF PROTO=TCP SPT=50990 DPT=443 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 11:52:08 fw-sede kernel: [UFW BLOCK] IN=eth1 OUT=eth0 MAC=52:54:00:1a:2b:3c:52:54:00:4d:5e:6f:08:00 SRC=10.20.36.30 DST=203.0.113.77 LEN=60 TOS=0x00 PREC=0x00 TTL=127 ID=40211 DF PROTO=TCP SPT=51007 DPT=443 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 10:09:51 fw-sede kernel: [UFW BLOCK] IN=eth0 OUT= MAC=52:54:00:aa:bb:cc:52:54:00:11:22:33:08:00 SRC=198.51.100.14 DST=10.20.32.10 LEN=60 TOS=0x00 PREC=0x00 TTL=49 ID=7001 DF PROTO=TCP SPT=41022 DPT=3389 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 10:09:51 fw-sede kernel: [UFW BLOCK] IN=eth0 OUT= MAC=52:54:00:aa:bb:cc:52:54:00:11:22:33:08:00 SRC=198.51.100.14 DST=10.20.32.11 LEN=60 TOS=0x00 PREC=0x00 TTL=49 ID=7002 DF PROTO=TCP SPT=41024 DPT=3389 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 10:09:52 fw-sede kernel: [UFW BLOCK] IN=eth0 OUT= MAC=52:54:00:aa:bb:cc:52:54:00:11:22:33:08:00 SRC=198.51.100.14 DST=10.20.36.5 LEN=60 TOS=0x00 PREC=0x00 TTL=49 ID=7003 DF PROTO=TCP SPT=41025 DPT=3389 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 13:41:19 fw-sede kernel: [UFW BLOCK] IN=eth2 OUT=eth0 MAC=52:54:00:dd:ee:ff:52:54:00:44:55:66:08:00 SRC=10.20.40.57 DST=203.0.113.77 LEN=52 TOS=0x00 PREC=0x00 TTL=127 ID=5120 DF PROTO=TCP SPT=49702 DPT=443 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 13:41:31 fw-sede kernel: [UFW BLOCK] IN=eth2 OUT=eth0 MAC=52:54:00:dd:ee:ff:52:54:00:44:55:66:08:00 SRC=10.20.40.57 DST=203.0.113.77 LEN=52 TOS=0x00 PREC=0x00 TTL=127 ID=5133 DF PROTO=TCP SPT=49705 DPT=443 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 13:41:55 fw-sede kernel: [UFW BLOCK] IN=eth2 OUT=eth0 MAC=52:54:00:dd:ee:ff:52:54:00:44:55:66:08:00 SRC=10.20.40.57 DST=203.0.113.78 LEN=52 TOS=0x00 PREC=0x00 TTL=127 ID=5161 DF PROTO=TCP SPT=49711 DPT=443 WINDOW=64240 RES=0x00 SYN URGP=0\nOct  5 14:05:12 fw-sede kernel: [UFW BLOCK] IN=eth3 OUT=eth0 MAC=52:54:00:77:88:99:52:54:00:aa:bb:01:08:00 SRC=172.16.5.40 DST=192.0.2.53 LEN=71 TOS=0x00 PREC=0x00 TTL=63 ID=12001 PROTO=UDP SPT=53001 DPT=53 LEN=51\n'
EMBUTIDO['plano-de-enderecos.csv'] = 'segmento,cidr,interface,conteudo\nServidores da sede,10.20.32.0/22,eth1,"internet banking interno, arquivos, resolvedor DNS 10.20.32.53"\nAtendimento da sede,10.20.36.0/24,eth1,estações de atendimento e ouvidoria\nAgência Centro (caixas),10.20.40.0/24,eth2,"notebooks dos caixas, ex.: SA-CX-07 = 10.20.40.57"\nWi-Fi de visitantes (agências),172.16.5.0/24,eth3,"celulares de cooperados em espera, só internet"\nInternet,(todo o resto),eth0,\n'
EMBUTIDO['acessos-ib-contestados.txt'] = '# Extrato do log do internet banking (fictício) entregue pela Serra Azul — 3 transferências Pix contestadas\n# horario_brt;conta_cooperado;ip_origem;porta_origem;evento\n2026-10-05 08:03:51;coop-A (fictício);198.51.100.230;40112;login ok\n2026-10-05 08:05:10;coop-A (fictício);198.51.100.230;40130;pix enviado\n2026-10-05 08:22:07;coop-B (fictício);198.51.100.230;51877;login ok\n2026-10-05 08:24:40;coop-B (fictício);198.51.100.230;51902;pix enviado\n2026-10-05 09:01:33;coop-C (fictício);203.0.113.150;-;login ok\n2026-10-05 09:02:58;coop-C (fictício);203.0.113.150;-;pix enviado\n'

def carregar(nome):
    """Devolve o texto do arquivo: da rede se der, senão da cópia embutida."""
    try:
        with urllib.request.urlopen(BASE + nome, timeout=5) as r:   # timeout: não travar a aula
            texto = r.read().decode("utf-8")
        if texto == EMBUTIDO[nome]:                                   # confere com a cópia local
            print(f"🌐 {nome}: baixado do GitHub (idêntico à cópia embutida)")
            return texto
        print(f"⚠️ {nome}: a versão do GitHub difere da embutida; usando a embutida (é a que os hashes do notebook esperam)")
        return EMBUTIDO[nome]
    except Exception as erro:                                         # sem rede, 404, proxy...
        print(f"📦 {nome}: usando a cópia embutida ({type(erro).__name__})")
        return EMBUTIDO[nome]

print(f"{len(EMBUTIDO)} arquivo(s) do caso disponíveis:", ", ".join(EMBUTIDO))

In [ ]:
import csv, io, re, hashlib

TEXTOS = {nome: carregar(nome) for nome in EMBUTIDO}
for nome, txt in TEXTOS.items():
    print(f"{hashlib.sha256(txt.encode()).hexdigest()}  {nome}")
print("Proveniência: o fw-sede grava hora LOCAL (BRT) sem fuso — confirmado pela TI por escrito.\n")

# O plano de endereços vira um dicionário segmento → rede
SEGMENTOS = {}
for linha in csv.DictReader(io.StringIO(TEXTOS["plano-de-enderecos.csv"])):
    if linha["cidr"].startswith("("):
        continue                                          # "Internet" é "todo o resto"
    SEGMENTOS[linha["segmento"]] = ipaddress.ip_network(linha["cidr"])
    print(f"{linha['segmento']:32} {linha['cidr']:15} {linha['interface']:5} {linha['conteudo']}")

def segmento(ip):
    a = ipaddress.ip_address(ip)
    return next((nome for nome, rede in SEGMENTOS.items() if a in rede), "Internet")

## 3. Do log bruto ao fluxo (Passos 2 e 3), com fuso

O `grep | sed` do original vira uma expressão regular. A hora do *syslog* não tem ano nem fuso: completamos com 2026 e BRT, como a TI informou, e convertemos para UTC.

In [ ]:
from datetime import datetime, timezone, timedelta

BRT = timezone(timedelta(hours=-3), "BRT")
PADRAO = re.compile(r"^(\w{3}\s+\d+ \d\d:\d\d:\d\d) (\S+) .*?\[UFW (\w+)\] IN=(\S*) OUT=(\S*) .*?"
                    r"SRC=(\S+) DST=(\S+) .*?TTL=(\d+) .*?PROTO=(\w+) SPT=(\d+) DPT=(\d+)")

EVENTOS = []
for linha in TEXTOS["ufw-fw-sede.log"].splitlines():
    m = PADRAO.search(linha)
    if not m:
        continue
    quando, fw, acao, i, o, src, dst, ttl, proto, spt, dpt = m.groups()
    local = datetime.strptime("2026 " + quando, "%Y %b %d %H:%M:%S").replace(tzinfo=BRT)
    EVENTOS.append({"utc": local.astimezone(timezone.utc), "brt": local, "acao": acao, "in": i, "out": o,
                    "src": src, "dst": dst, "ttl": int(ttl), "proto": proto, "dpt": int(dpt),
                    "seg_src": segmento(src), "seg_dst": segmento(dst)})

for e in EVENTOS:
    print(f"{e['brt']:%H:%M:%S} BRT = {e['utc']:%H:%M:%S}Z  {e['src']:15} -> {e['dst']:15} "
          f"{e['proto']}/{e['dpt']:<5} {e['acao']:5} [{e['seg_src']} → {e['seg_dst']}]")

### Cruzando com o `conn.log` do Dia 4

As `S0` perto do meio-dia e a varredura RDP precisam aparecer nos dois logs, no mesmo instante UTC. Se aparecessem com 3 horas de diferença, o erro estaria no fuso, não no caso.

In [ ]:
CONN_EMBUTIDO = '#separator \\x09\n#set_separator\t,\n#empty_field\t(empty)\n#unset_field\t-\n#path\tconn\n#open\t2026-10-05-15-10-00\n#fields\tts\tuid\tid.orig_h\tid.orig_p\tid.resp_h\tid.resp_p\tproto\tservice\tduration\torig_bytes\tresp_bytes\tconn_state\thistory\n#types\ttime\tstring\taddr\tport\taddr\tport\tenum\tstring\tinterval\tcount\tcount\tstring\tstring\n1791198842.118000\tCsA1kq3\t10.20.36.21\t53110\t10.20.32.53\t53\tudp\tdns\t0.002\t39\t55\tSF\tDd\n1791198842.131000\tCsA1kq4\t10.20.36.21\t50214\t203.0.113.77\t443\ttcp\tssl\t48.912\t1873\t41250\tSF\tShADadFf\n1791199905.402000\tCsA1kq9\t10.20.36.44\t50388\t203.0.113.77\t443\ttcp\tssl\t61.004\t2290\t41377\tSF\tShADadFf\n1791201737.006000\tCsA1kr2\t10.20.36.44\t50412\t203.0.113.77\t443\ttcp\tssl\t0.611\t517\t0\tRSTR\tShADr\n1791205791.550000\tCsA1kt0\t198.51.100.14\t41022\t10.20.32.10\t3389\ttcp\t-\t0.000\t0\t0\tS0\tS\n1791205791.601000\tCsA1kt1\t198.51.100.14\t41024\t10.20.32.11\t3389\ttcp\t-\t0.000\t0\t0\tS0\tS\n1791211653.200000\tCsA1ku5\t10.20.36.21\t50990\t203.0.113.77\t443\ttcp\t-\t9.012\t0\t0\tS0\tS\n1791211928.900000\tCsA1ku8\t10.20.36.30\t51007\t203.0.113.77\t443\ttcp\t-\t9.008\t0\t0\tS0\tS\n1791212620.000000\tCsA1kv0\t10.20.36.30\t51102\t10.20.32.80\t443\ttcp\tssl\t0.402\t1210\t5320\tSF\tShADadFf\n#close\t2026-10-05-15-10-00\n'
try:
    with urllib.request.urlopen(BASE.replace("dia-09", "dia-04") + "conn.log", timeout=5) as r:
        conn_txt = r.read().decode()
except Exception:
    conn_txt = CONN_EMBUTIDO

campos, conn = None, []
for l in conn_txt.splitlines():
    if l.startswith("#fields"):
        campos = l.split("\t")[1:]
    elif l and not l.startswith("#"):
        c = dict(zip(campos, l.split("\t")))
        c["utc"] = datetime.fromtimestamp(float(c["ts"]), timezone.utc)
        conn.append(c)

for e in EVENTOS:
    par = [c for c in conn if c["id.orig_h"] == e["src"] and c["id.resp_h"] == e["dst"]
           and abs((c["utc"] - e["utc"]).total_seconds()) < 60]          # mesmo par, mesmo minuto
    if par:
        c = par[0]
        print(f"✅ fw {e['utc']:%H:%M:%S}Z {e['src']}→{e['dst']}:{e['dpt']} BLOCK  ⇔  conn.log {c['utc']:%H:%M:%S}Z {c['conn_state']} {c['history']}")
print("\n➡️ As S0 do Dia 4 eram o firewall descartando o SYN (o sensor fica antes do firewall).")
print("   O que o fw-sede tem e o conn.log NÃO tem: a Agência Centro (o sensor só olha a sede).")

## 4. Prioridade por direção e segmento (Passo 4)

Uma regra **escrita antes** de ler as linhas (Camada 8 do README). Você pode mudar os pesos, mas mude aqui, explicitamente.

In [ ]:
from collections import defaultdict

PORTAS_SENSIVEIS = {22, 3389, 4444, 445}
INFRA_DO_CASO = {"203.0.113.77", "203.0.113.78"}

def prioridade(fluxo):
    """Devolve (nota, motivo). Quanto maior a nota, mais prioritário."""
    src_interno = fluxo["seg_src"] != "Internet"
    if src_interno and fluxo["dst"] in INFRA_DO_CASO and fluxo["seg_src"].startswith("Agência"):
        return 3, "SAÍDA de segmento sensível (caixas) para a infraestrutura do golpe — fora do padrão da sede"
    if src_interno and fluxo["dst"] in INFRA_DO_CASO:
        return 2, "saída do atendimento para o golpe; compatível com verificação das denúncias, já bloqueada pela TI"
    if not src_interno and fluxo["dpt"] in PORTAS_SENSIVEIS:
        return 1, "entrada vinda da internet em porta sensível: varredura, ruído de fundo"
    return 0, "provável bloqueio de política"

fluxos = defaultdict(list)
for e in EVENTOS:
    fluxos[(e["src"], e["dst"], e["proto"], e["dpt"])].append(e)

RESUMO = []
for (src, dst, proto, dpt), evs in fluxos.items():
    nota, motivo = prioridade(evs[0])
    RESUMO.append((nota, src, dst, proto, dpt, len(evs), evs[0]["utc"], evs[-1]["utc"], evs[0], motivo))
RESUMO.sort(key=lambda x: (-x[0], x[6]))

for nota, src, dst, proto, dpt, n, t0, t1, e, motivo in RESUMO:
    print(f"[{'🔴' if nota == 3 else '🟠' if nota == 2 else '🟡' if nota == 1 else '⚪'}] "
          f"{e['seg_src']} {src} → {e['seg_dst']} {dst} {proto}/{dpt} ×{n}, BLOCK, {t0:%H:%M:%S}–{t1:%H:%M:%S}Z")
    print(f"      {motivo}")
e = next(x for x in EVENTOS if x["src"] == "10.20.40.57")
print(f"\nTTL {e['ttl']} nos pacotes de 10.20.40.57 → combina com Windows (128) atravessando 1 salto (INFERÊNCIA).")

## 5. O IP do internet banking: o que ele prova?

In [ ]:
print(TEXTOS["acessos-ib-contestados.txt"])
por_ip = defaultdict(list)
for l in TEXTOS["acessos-ib-contestados.txt"].splitlines():
    if l.startswith("#"):
        continue
    hora, conta, ip, porta, evento = l.split(";")
    por_ip[ip].append((hora, conta, porta, evento))

for ip, linhas in por_ip.items():
    a = ipaddress.ip_address(ip)
    contas = sorted({c for _, c, _, _ in linhas})
    print(f"{ip}: {len(contas)} conta(s) {contas}; portas {[p for _, _, p, _ in linhas]}")
    print(f"   RFC1918={any(a in r for r in RFC1918)}  CGNAT(100.64/10)={a in CGNAT}  documentação={a.is_private and not any(a in r for r in RFC1918)}")

HIPOTESES = [
    ("Mesmo operador do golpe usando a mesma conexão", "padrão de horário/dispositivo no IB; mesma impressão digital do navegador"),
    ("NAT de operadora móvel: muitos assinantes atrás do mesmo IP", "RDAP do bloco (é operadora?); porta + horário exato para a operadora (só por ordem judicial)"),
    ("VPN ou proxy comercial compartilhado", "RDAP/ASN do bloco (é provedor de VPN/hosting?); outros acessos do mesmo IP no IB"),
]
print("\nHipóteses para 198.51.100.230 (coop-A e coop-B):")
for h, separa in HIPOTESES:
    print(f"  • {h}\n      o que separaria: {separa}")
print("\ncoop-C sem porta de origem: atrás de NAT, IP sem porta + horário pode não apontar ninguém.")

## 6. 📝 O mapa de fluxos e os achados

In [ ]:
from pathlib import Path

ids = {}
def nid(ip):
    return ids.setdefault(ip, "H" + ip.replace(".", "_"))

grupos = defaultdict(set)
for _, src, dst, *_ in RESUMO:
    grupos[segmento(src)].add(src); grupos[segmento(dst)].add(dst)

linhas = ["flowchart LR"]
for g, (seg, ips) in enumerate(grupos.items()):
    rede = SEGMENTOS.get(seg)
    titulo = f"{seg} {rede}" if rede else seg
    linhas.append(f'    subgraph SEG{g}["{titulo}"]')
    linhas += [f'        {nid(ip)}["{ip}"]' for ip in sorted(ips)]
    linhas.append("    end")
linhas.append('    %% Todas as setas atravessam o fw-sede (eth0 internet, eth1 sede, eth2 Agência Centro, eth3 Wi-Fi visitantes)')
vermelhas = []
for k, (nota, src, dst, proto, dpt, n, t0, t1, e, motivo) in enumerate(RESUMO):
    rot = f"{proto}/{dpt} x{n}, BLOCK, {t0:%H:%M:%S} a {t1:%H:%M:%S}Z"
    if nota == 3:
        rot += "<br/>PRIORIDADE: caixa da Agência Centro → infraestrutura do golpe"
        vermelhas.append(str(k))
    linhas.append(f'    {nid(src)} -->|"{rot}"| {nid(dst)}')
if vermelhas:
    linhas.append(f"    linkStyle {','.join(vermelhas)} stroke:#d62728,stroke-width:3px")
MAPA = "\n".join(linhas)

achados = ["# Achados — firewall da sede (Operação Isca)", "",
           "Fonte: ufw-fw-sede.log (hora local BRT, sem fuso; confirmado pela TI). Horários abaixo em UTC.", ""]
for nota, src, dst, proto, dpt, n, t0, t1, e, motivo in RESUMO:
    achados.append(f"- **Prioridade {nota}** · {e['seg_src']} ({src}) → {e['seg_dst']} ({dst}) {proto}/{dpt} ×{n}, BLOCK, "
                   f"{t0:%H:%M:%S}–{t1:%H:%M:%S}Z · {motivo}")
achados += ["",
            "- 10.20.40.57 (Agência Centro): pista prioritária, NÃO concluída. Falta: o que rodava na máquina naquele minuto "
            "(processos/conexões do host), se houve conexão por outro caminho, e o contexto operacional. "
            "Este log descreve um ponto de rede num horário; não identifica quem usava a máquina. "
            "O UFW aqui só registra BLOQUEIOS: a ausência de linhas antes de 16:41Z não prova ausência de conexão "
            "antes do bloqueio (pedir o log de conexões permitidas).",
            "- 198.51.100.230 (IB, coop-A e coop-B): três hipóteses abertas (mesmo operador, CGNAT, VPN/proxy). "
            "Próximo passo lícito: RDAP do bloco. Identificação de assinante: só por ordem judicial, via autoridade."]
Path("notas").mkdir(exist_ok=True)
Path("notas/day09-segment-flows-isca.md").write_text("```mermaid\n" + MAPA + "\n```\n", encoding="utf-8")
Path("notas/day09-findings-isca.md").write_text("\n".join(achados) + "\n", encoding="utf-8")
print(MAPA)
print()
print("\n".join(achados))
print("\n💾 notas/day09-segment-flows-isca.md · notas/day09-findings-isca.md")
try:
    from google.colab import files
    files.download("notas/day09-segment-flows-isca.md"); files.download("notas/day09-findings-isca.md")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta notas/)")

## ✅ Antes de fechar o Dia 9

- [ ] Pelo menos 8/10 no jogo do tamanho de bloco
- [ ] Sabe por que não confiar em `is_private`
- [ ] Fuso do firewall registrado na custódia; `S0` do Dia 4 explicadas com fonte
- [ ] `10.20.40.57` registrado como pista prioritária, sem conclusão e sem pessoa
- [ ] Três hipóteses para o IP compartilhado do internet banking
- [ ] Fez também a prática original (`ufw-sample.log`, `10.20.34.19`, `172.20.14.77/20`)

**Amanhã (Dia 10):** o shell Linux, o sistema de arquivos e as permissões. Os timestamps de um arquivo também mentem, e você vai aprender qual deles não consegue.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>